<a href="https://colab.research.google.com/github/raki-rankawat/thesis-v1/blob/master/Model_Pruning_Combined.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model_Pruning_Combined
### Unstructured + Structured pruning sweep — MobileNetV2 & MobileNetV3

**Experiments:**
- **Unstructured** (L1, weight-level): 10 / 20 / 30%
- **Structured** (L2 filter, pointwise only): 5 / 10 / 15%

**Per-run pipeline:**
1. Load seed checkpoint → prune → eval (no FT) → fine-tune → save `.pth`
2. Export FP32 ONNX → INT8 QDQ → compute NSE → flag quant-compatibility
3. Count non-zero parameters (correct for structured, exact for unstructured)

Checkpoints that **fail** the NSE gate (`< 0.95`) are flagged and **must not** be passed to `Pipeline_Quantz`.

**Resume support:** progress is saved to Drive after every experiment (`pruning_combined_records.json`).
Re-running the sweep cell skips already-completed experiments automatically.

> **Previously done:** unstructured 10/20/30% for both models ✅  
> **New this run:** structured 5/10/15%


In [1]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/stm32-thesis/utils/")

Mounted at /content/drive
✅ utils loaded from Drive


In [2]:
# ── Imports ──────────────────────────────────────────────────────────
import os, time, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune

from pathlib import Path

from utils.dataset import prepare_dataset, get_loaders, get_test_loader
from utils.models  import VWW_MobileNetV2, VWW_MobileNetV3
from utils.train   import setup_device, set_seed, evaluate, train_epoch, validate_epoch

device = setup_device(seed=41)

Device: cuda


In [3]:
# ── Dataset ──────────────────────────────────────────────────────────
prepare_dataset()
train_loader, val_loader = get_loaders(batch_size=64, augmentation="standard")
test_loader              = get_test_loader(batch_size=1)
print("✅ Dataset ready")

1/4 Download
⬇️  Downloading VWW archive...
✅ Download complete: /content/vww_work/vw_coco2014_96.tar.gz
2/4 Extract
📦 Extracting VWW archive...
✅ Extraction complete: /content/vww_work/extracted
3/4 Find root
   Root: /content/vww_work/extracted/vw_coco2014_96
4/4 Manifests
✅ Manifests already exist: /content/drive/My Drive/vww_fixed_split_manifests
Train: 7000 | Val: 1500 | Batch: 64
Test: 1500 samples  ⚠️  Use only for final evaluation
✅ Dataset ready


In [4]:
# ── Config — edit here ───────────────────────────────────────────────
SAVE_DIR   = "/content/drive/My Drive/stm32-thesis/checkpoints"
EXPORT_DIR = Path("/content/drive/My Drive/stm32-thesis/exports")
SHARED_DIR = EXPORT_DIR / "shared"

# Input checkpoints — best seed from training runs
CKPTS = {
    "MobileNetV2": f"{SAVE_DIR}/mobilenetv2_seed_74.pth",
    "MobileNetV3": f"{SAVE_DIR}/mobilenetv3_seed_74.pth",
}

MODEL_FNS = {
    "MobileNetV2": VWW_MobileNetV2,
    "MobileNetV3": VWW_MobileNetV3,
}

# ── Sweep config ──────────────────────────────────────────────────────
ACTIVE_MODELS = ["MobileNetV2", "MobileNetV3"]

# Unstructured: 10/20/30
UNSTRUCTURED_AMOUNTS = [0.10, 0.20, 0.30]   # 10%, 20%, 30%

# Structured: 5/10/15
STRUCTURED_AMOUNTS   = [0.05, 0.10, 0.15]  # 5%, 10%, 15%

# ── Fine-tune config ──────────────────────────────────────────────────
FT_EPOCHS_UNSTRUCT = 10   # same as original sweep
FT_EPOCHS_STRUCT   = 15   # structured needs more recovery time
FT_LR  = 1e-4
FT_WD  = 1e-4

# ── Quant gate ────────────────────────────────────────────────────────
NSE_THRESHOLD = 0.95   # below this → ⛔ do not pass to Pipeline_Quantz

print(f"Unstructured amounts : {[f'{int(a*100)}%' for a in UNSTRUCTURED_AMOUNTS]}")
print(f"Structured amounts   : {[f'{int(a*100)}%' for a in STRUCTURED_AMOUNTS]}")
print(f"FT epochs (unstruct) : {FT_EPOCHS_UNSTRUCT}")
print(f"FT epochs (struct)   : {FT_EPOCHS_STRUCT}")
print(f"NSE threshold        : {NSE_THRESHOLD}")

Unstructured amounts : ['10%', '20%', '30%']
Structured amounts   : ['5%', '10%', '15%']
FT epochs (unstruct) : 10
FT epochs (struct)   : 15
NSE threshold        : 0.95


In [5]:
# ── Pruning helpers ──────────────────────────────────────────────────

def apply_unstructured(model, amount):
    """L1 unstructured: zeros individual weights across ALL Conv2d layers."""
    params = [(m, "weight") for m in model.modules() if isinstance(m, nn.Conv2d)]
    for layer, p in params:
        prune.l1_unstructured(layer, name=p, amount=amount)
    return params


def apply_structured(model, amount):
    """
    L2 structured filter pruning on pointwise Conv2d layers only.
    Depthwise layers (groups == in_channels) are skipped to preserve
    channel alignment inside inverted-residual blocks.
    Returns list of (layer, param_name) for mask removal.
    """
    params = []
    for m in model.modules():
        if isinstance(m, nn.Conv2d) and m.groups == 1:
            prune.ln_structured(m, name="weight", amount=amount, n=2, dim=0)
            params.append((m, "weight"))
    return params


def remove_masks(params):
    """Bake pruning masks permanently into weights before saving / fine-tuning."""
    for layer, p in params:
        prune.remove(layer, p)


def compute_sparsity(model):
    """Fraction of zero weights across all Conv2d layers (weight-level)."""
    total = zeroed = 0
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            w = m.weight.detach().cpu().numpy()
            total  += w.size
            zeroed += (w == 0).sum()
    return zeroed / total if total > 0 else 0.0


def count_params_total(model):
    """Total parameter count (all tensors, including zero-padded structured weights)."""
    return sum(p.numel() for p in model.parameters())


def count_params_nonzero(model):
    """
    Non-zero parameter count — mask-aware.
    PyTorch pruning stores the original weights in weight_orig and applies
    weight_mask as a forward hook. model.parameters() returns weight_orig,
    NOT the effective masked weight — so a naive (p != 0).sum() always returns
    the full count while masks are active.
    This function checks weight_orig * weight_mask for pruned layers so the
    count reflects the true effective (non-zeroed) parameters.
    """
    total = 0
    visited = set()
    for name, buf in model.named_buffers():
        # weight_mask buffers are named like 'layer.weight_mask'
        if name.endswith('_mask'):
            param_name = name[:-5]  # strip '_mask' → 'layer.weight'
            orig_name  = param_name + '_orig'
            # get weight_orig
            orig = dict(model.named_parameters()).get(orig_name, None)
            if orig is not None:
                effective = orig * buf
                total += (effective != 0).sum().item()
                visited.add(orig_name)
    # Add non-pruned parameters normally
    for pname, p in model.named_parameters():
        if pname not in visited:
            total += (p != 0).sum().item()
    return total


print("✅ Pruning helpers loaded")

✅ Pruning helpers loaded


In [6]:
# ── Fine-tune helper ─────────────────────────────────────────────────

def fine_tune(model, epochs, lr=FT_LR, wd=FT_WD):
    set_seed(41)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_acc   = 0.0
    best_state = None

    for epoch in range(1, epochs + 1):
        _, ta = train_epoch(model, train_loader, optimizer, criterion, device)
        _, va = validate_epoch(model, val_loader, criterion, device)
        scheduler.step()

        marker = " ✅" if va > best_acc else ""
        print(f"  FT {epoch:2d}/{epochs} | train {ta*100:.2f}% | val {va*100:.2f}%{marker}")

        if va > best_acc:
            best_acc   = va
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_acc


print("✅ Fine-tune helper loaded")

✅ Fine-tune helper loaded


In [7]:
# ── Quantisation helpers ─────────────────────────────────────────────
!pip -q install onnx onnxruntime onnxruntime-tools

import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_static, QuantType, QuantFormat, CalibrationDataReader


def export_onnx(model, path):
    if Path(path).exists():
        print(f"    ⏭️  FP32 ONNX exists"); return
    model.eval()
    dummy = torch.randn(1, 3, 96, 96, device=device)
    torch.onnx.export(
        model, dummy, str(path),
        input_names=["input"], output_names=["logits"],
        export_params=True, opset_version=18,
        do_constant_folding=True,
        dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},
        dynamo=False,
    )
    onnx.checker.check_model(str(path), full_check=False)
    print(f"    ✅ FP32 ONNX saved")


def save_calib_npz(path, n=200):
    if Path(path).exists():
        print(f"    ⏭️  Calib data exists"); return
    xs = []
    with torch.no_grad():
        for i, (x, _) in enumerate(train_loader):
            if i >= n: break
            xs.append(x.numpy().astype("float32")[0])
    np.savez(path, input=np.stack(xs))
    print(f"    ✅ Calib data saved ({n} samples)")


def generate_shared_test_files(n=200):
    SHARED_DIR.mkdir(parents=True, exist_ok=True)
    inp_p = SHARED_DIR / "test_input.npz"
    lbl_p = SHARED_DIR / "test_labels.npz"
    if inp_p.exists() and lbl_p.exists():
        print(f"    ⏭️  Shared test files exist")
        return inp_p, lbl_p
    inputs, labels = [], []
    for i, (x, y) in enumerate(test_loader):
        if i >= n: break
        inputs.append(x.numpy().astype("float32")[0])
        labels.append(int(y.item()))
    np.savez(inp_p, input=np.stack(inputs))
    np.savez(lbl_p, label=np.array(labels, dtype="int32"))
    print(f"    ✅ Shared test files saved ({n} samples)")
    return inp_p, lbl_p


class CalibReader(CalibrationDataReader):
    def __init__(self, npz_path):
        self.data = np.load(npz_path)["input"].astype("float32")
        self.i = 0
    def get_next(self):
        if self.i >= len(self.data): return None
        out = {"input": self.data[self.i:self.i+1]}; self.i += 1; return out
    def rewind(self): self.i = 0


def quantize_int8(fp32_path, calib_path, int8_path):
    if Path(int8_path).exists():
        print(f"    ⏭️  INT8 ONNX exists"); return
    quantize_static(
        model_input=str(fp32_path),
        model_output=str(int8_path),
        calibration_data_reader=CalibReader(calib_path),
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QInt8,
        weight_type=QuantType.QInt8,
        per_channel=True,
    )
    print(f"    ✅ INT8 QDQ ONNX saved")


def compute_nse(fp32_path, int8_path, input_npz):
    """
    Normalised Signal Energy: 1 - ||fp32 - int8||² / ||fp32 - mean(fp32)||²
    NSE ≥ 0.95 means quantisation distorts logits <5% — safe to deploy.
    """
    inputs = np.load(input_npz)["input"]
    s32 = ort.InferenceSession(str(fp32_path), providers=["CPUExecutionProvider"])
    s8  = ort.InferenceSession(str(int8_path), providers=["CPUExecutionProvider"])
    fp32_outs, int8_outs = [], []
    for i in range(len(inputs)):
        sample = inputs[i:i+1]
        fp32_outs.append(s32.run(["logits"], {"input": sample})[0][0])
        int8_outs.append(s8.run(["logits"],  {"input": sample})[0][0])
    fp32_outs = np.array(fp32_outs)
    int8_outs = np.array(int8_outs)
    num = np.sum((fp32_outs - int8_outs) ** 2)
    den = np.sum((fp32_outs - fp32_outs.mean()) ** 2)
    return float(1 - num / den)


print("✅ Quantisation helpers loaded")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.7/212.7 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.1 MB/s eta 0:00:00
✅ Quantisation helpers loaded


In [8]:
# ── Pre-flight ───────────────────────────────────────────────────────
print("Checking seed checkpoints...")
for model_name, ckpt_path in CKPTS.items():
    ok = os.path.exists(ckpt_path)
    print(f"  {'✅' if ok else '❌'}  {model_name}: {ckpt_path}")
    if not ok:
        raise FileNotFoundError(f"Missing seed checkpoint: {ckpt_path}")

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
SHARED_INPUT_NPZ, _ = generate_shared_test_files()
print("\n✅ Pre-flight passed")

Checking seed checkpoints...
  ✅  MobileNetV2: /content/drive/My Drive/stm32-thesis/checkpoints/mobilenetv2_seed_74.pth
  ✅  MobileNetV3: /content/drive/My Drive/stm32-thesis/checkpoints/mobilenetv3_seed_74.pth
    ⏭️  Shared test files exist

✅ Pre-flight passed


In [ ]:
# ── Consolidated pruning sweep ───────────────────────────────────────
# 2 models × (3 unstructured + 3 structured) = 12 experiments total.
# Each run: load → prune → eval → fine-tune → save .pth → ONNX → INT8 → NSE gate.
#
# ── Resume support ───────────────────────────────────────────────────
# Progress is saved to Drive after every experiment as a JSON.
# Re-running this cell skips already-completed experiments automatically.
# To force a re-run of a specific experiment, delete its entry from the JSON
# or delete the JSON entirely to restart from scratch.

import json as _json
import math

RECORDS_JSON = f"{SAVE_DIR}/pruning_combined_records.json"

# ── Load existing progress ────────────────────────────────────────────
if os.path.exists(RECORDS_JSON):
    with open(RECORDS_JSON) as _f:
        records = _json.load(_f)
    print(f"✅ Loaded {len(records)} existing records from Drive")
    for _r in records:
        print(f"   ↩️  {_r['model']} | {_r['prune_type']:12s} | {_r['target_%']:2d}%  "
              f"→ after_FT: {_r['after_FT_%']:.2f}%  quant_ok: {_r['quant_ok']}")
else:
    records = []
    print("No existing records found — starting fresh")

print()

# ── Helper: check if experiment already completed ─────────────────────
def _already_done(model_name, prune_type, pct):
    return any(
        r["model"] == model_name
        and r["prune_type"] == prune_type
        and r["target_%"] == pct
        for r in records
    )

for model_name in ACTIVE_MODELS:
    model_fn  = MODEL_FNS[model_name]
    ckpt_path = CKPTS[model_name]

    # ── Baseline (load once per architecture) ───────────────────────
    _base = model_fn().to(device)
    _base.load_state_dict(torch.load(ckpt_path, map_location=device))
    baseline_acc    = evaluate(_base, val_loader, device)
    baseline_params = count_params_total(_base)
    baseline_nz     = count_params_nonzero(_base)
    del _base

    print(f"\n{'═'*65}")
    print(f"  {model_name}  —  baseline: {baseline_acc*100:.2f}%  "
          f"params: {baseline_params:,}  non-zero: {baseline_nz:,}")
    print(f"{'═'*65}")

    # ── Build experiment list: (prune_type, amount, ft_epochs) ──────
    experiments = (
        [("unstructured", a, FT_EPOCHS_UNSTRUCT) for a in UNSTRUCTURED_AMOUNTS]
      + [("structured",   a, FT_EPOCHS_STRUCT)   for a in STRUCTURED_AMOUNTS]
    )

    for prune_type, amount, ft_epochs in experiments:
        pct = int(amount * 100)
        tag = f"{model_name.lower()}_{prune_type}_{pct}pct"

        # ── Skip if already done ─────────────────────────────────────
        if _already_done(model_name, prune_type, pct):
            print(f"\n  ⏭️  {model_name} | {prune_type:12s} | {pct:2d}%  — already done, skipping")
            continue

        print(f"\n▶ {model_name} | {prune_type:12s} | {pct:2d}%")

        # Output paths
        out_pth    = f"{SAVE_DIR}/{tag}.pth"
        out_dir    = EXPORT_DIR / tag
        out_dir.mkdir(parents=True, exist_ok=True)
        fp32_path  = out_dir / "model_fp32.onnx"
        calib_path = out_dir / "calib_train.npz"
        int8_path  = out_dir / "model_int8_qdq.onnx"

        # ── Load fresh weights every run ────────────────────────────
        model = model_fn().to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))

        # ── Prune ───────────────────────────────────────────────────
        if prune_type == "unstructured":
            applied = apply_unstructured(model, amount)
        else:
            applied = apply_structured(model, amount)

        post_prune_acc  = evaluate(model, val_loader, device)
        actual_sparsity = compute_sparsity(model)
        drop = (post_prune_acc - baseline_acc) * 100
        print(f"  After prune  : {post_prune_acc*100:.2f}%  "
              f"drop: {drop:+.2f}%  sparsity: {actual_sparsity*100:.1f}%")

        # ── Param counts (measured on pruned model, before FT) ──────
        params_total     = count_params_total(model)
        params_nonzero   = count_params_nonzero(model)
        nz_reduction_pct = (1 - params_nonzero / baseline_nz) * 100
        print(f"  Params total : {params_total:,}  "
              f"non-zero: {params_nonzero:,}  "
              f"(effective -{nz_reduction_pct:.1f}% vs baseline)")

        # ── Fine-tune (masks must stay ON during FT) ─────────────────
        # Masks keep pruned weights zeroed throughout training so the
        # optimizer cannot refill them. Remove only after FT is done.
        t0     = time.time()
        ft_acc = fine_tune(model, ft_epochs)
        elapsed = (time.time() - t0) / 60
        ft_delta = (ft_acc - baseline_acc) * 100
        print(f"  After FT     : {ft_acc*100:.2f}%  "
              f"delta vs baseline: {ft_delta:+.2f}%  ({elapsed:.1f} min)")

        # Bake masks permanently AFTER fine-tune, then save
        remove_masks(applied)
        torch.save(model.state_dict(), out_pth)
        print(f"  Checkpoint   : {tag}.pth")

        # ── Quantisation ─────────────────────────────────────────────
        model.eval()
        export_onnx(model, fp32_path)
        save_calib_npz(calib_path)
        try:
            quantize_int8(fp32_path, calib_path, int8_path)
            nse    = compute_nse(fp32_path, int8_path, SHARED_INPUT_NPZ)
            nse_ok = nse >= NSE_THRESHOLD
            quant_error = None
        except Exception as e:
            nse    = float("nan")
            nse_ok = False
            quant_error = str(e)
            print(f"    ⚠️  Quantisation failed: {e}")

        status = "✅ deploy" if nse_ok else "⛔ skip Pipeline_Quantz"
        print(f"  NSE          : {nse:.4f}  {status}")

        records.append({
            "model"           : model_name,
            "prune_type"      : prune_type,
            "target_%"        : pct,
            "sparsity_%"      : round(actual_sparsity * 100, 1),
            "baseline_%"      : round(baseline_acc * 100, 2),
            "post_prune_%"    : round(post_prune_acc * 100, 2),
            "drop_%"          : round(drop, 2),
            "after_FT_%"      : round(ft_acc * 100, 2),
            "delta_%"         : round(ft_delta, 2),
            "params_total"    : params_total,
            "params_nonzero"  : params_nonzero,
            "nz_reduction_%"  : round(nz_reduction_pct, 1),
            "NSE"             : round(nse, 4) if not math.isnan(nse) else None,
            "quant_ok"        : nse_ok,
            "quant_error"     : quant_error,
            "ckpt"            : out_pth,
            "int8_path"       : str(int8_path) if nse_ok else None,
        })

        # ── Save progress to Drive immediately ───────────────────────
        with open(RECORDS_JSON, "w") as _f:
            _json.dump(records, _f, indent=2)
        print(f"  💾 Progress saved ({len(records)} records total)")

        del model

print("\n\n✅ Full sweep complete (or all already done).")
print(f"   Records JSON: {RECORDS_JSON}")



═════════════════════════════════════════════════════════════════
  MobileNetV2  —  baseline: 78.40%  params: 151,874  non-zero: 151,874
═════════════════════════════════════════════════════════════════

▶ MobileNetV2 | unstructured | 10%
  After prune  : 78.47%  drop: +0.07%  sparsity: 10.0%
  Params total : 151,874  non-zero: 137,421  (effective -9.5% vs baseline)
  FT  1/10 | train 81.44% | val 78.27% ✅
  FT  2/10 | train 82.06% | val 78.33% ✅
  FT  3/10 | train 81.57% | val 78.27%
  FT  4/10 | train 81.60% | val 78.20%
  FT  5/10 | train 82.21% | val 78.60% ✅
  FT  6/10 | train 81.91% | val 78.60%
  FT  7/10 | train 82.17% | val 79.07% ✅
  FT  8/10 | train 82.66% | val 78.67%
  FT  9/10 | train 82.23% | val 78.80%
  FT 10/10 | train 82.73% | val 78.80%
  After FT     : 79.07%  delta vs baseline: +0.67%  (2.3 min)
  Checkpoint   : mobilenetv2_unstructured_10pct.pth
    ⏭️  FP32 ONNX exists
    ⏭️  Calib data exists
    ⏭️  INT8 ONNX exists


In [ ]:
# ── Results table ────────────────────────────────────────────────────
df = pd.DataFrame(records)
df = df.sort_values(["model", "prune_type", "target_%"]).reset_index(drop=True)

print("\n" + "="*105)
print("CONSOLIDATED PRUNING SWEEP RESULTS")
print("="*105)

display_cols = [
    "model", "prune_type", "target_%", "sparsity_%",
    "baseline_%", "post_prune_%", "drop_%",
    "after_FT_%", "delta_%", "NSE", "quant_ok",
]
print(df[display_cols].to_string(index=False))
print("="*105)

# ── Parameter analysis (structured only — unstructured has same total count) ──
print("\nParameter reduction (non-zero counts):")
print(f"  {'Model':<14} {'Type':<13} {'%':>4}  {'Baseline':>10}  {'Non-zero':>10}  {'Reduction':>10}")
print(f"  {'-'*65}")
for _, row in df.iterrows():
    print(f"  {row['model']:<14} {row['prune_type']:<13} {int(row['target_%']):>3}%  "
          f"{row['params_total']:>10,}  {row['params_nonzero']:>10,}  "
          f"{row['nz_reduction_%']:>9.1f}%")

# ── Quant gate summary ────────────────────────────────────────────────────────
print("\nQuantisation gate (NSE ≥ 0.95):")
for _, row in df.iterrows():
    gate = "✅ PASS — safe for Pipeline_Quantz" if row["quant_ok"] else "⛔ FAIL — exclude from Pipeline_Quantz"
    nse_str = f"{row['NSE']:.4f}" if row["NSE"] is not None else "ERROR"
    print(f"  {row['model']:<14} {row['prune_type']:<13} {int(row['target_%']):>3}%  "
          f"NSE={nse_str}  {gate}")

# ── Best per category ─────────────────────────────────────────────────────────
print("\n🏆 Best after FT (quant-compatible only):")
deployable = df[df["quant_ok"] == True]
if len(deployable) > 0:
    for model_name in df["model"].unique():
        sub = deployable[deployable["model"] == model_name]
        if sub.empty: continue
        for ptype in ["unstructured", "structured"]:
            sub2 = sub[sub["prune_type"] == ptype]
            if sub2.empty: continue
            best = sub2.loc[sub2["after_FT_%"].idxmax()]
            print(f"  {model_name} {ptype:12s}: {int(best['target_%'])}%  "
                  f"→ {best['after_FT_%']:.2f}%  (delta {best['delta_%']:+.2f}%  "
                  f"NSE={best['NSE']:.4f})")
else:
    print("  ⚠️  No quant-compatible checkpoints found.")

# ── Export deployable checkpoint list ────────────────────────────────────────
deployable_ckpts = df[df["quant_ok"] == True][["model", "prune_type", "target_%", "after_FT_%", "NSE", "int8_path", "ckpt"]]
print("\n📋 Deployable checkpoints (safe to pass to Pipeline_Quantz):")
print(deployable_ckpts.to_string(index=False))

In [ ]:
# ── Damage curves — accuracy vs pruning amount by type & model ───────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("Pruning Damage Curves — MobileNetV2 & MobileNetV3", fontsize=14, fontweight="bold")

for row_idx, model_name in enumerate(ACTIVE_MODELS):
    for col_idx, prune_type in enumerate(["unstructured", "structured"]):
        ax  = axes[row_idx][col_idx]
        sub = df[(df["model"] == model_name) & (df["prune_type"] == prune_type)].sort_values("target_%")

        if sub.empty:
            ax.set_visible(False); continue

        baseline = sub["baseline_%"].iloc[0]
        amounts  = sub["target_%"].tolist()

        ax.axhline(baseline, color="gray", linestyle="--", linewidth=1.2,
                   label=f"Baseline {baseline:.2f}%")
        ax.plot(amounts, sub["post_prune_%"].tolist(), "o--", color="tomato",
                label="After prune (no FT)")
        ax.plot(amounts, sub["after_FT_%"].tolist(), "s-", color="steelblue",
                label="After FT")

        # Shade quant-failed runs
        failed = sub[sub["quant_ok"] == False]
        for _, row in failed.iterrows():
            ax.axvspan(row["target_%"] - 0.3, row["target_%"] + 0.3,
                       alpha=0.2, color="red", label="_nolegend_")

        # Mark NSE values above each point
        for _, row in sub.iterrows():
            nse_str = f"{row['NSE']:.3f}" if row["NSE"] is not None else "err"
            color   = "green" if row["quant_ok"] else "red"
            ax.annotate(f"NSE={nse_str}", (row["target_%"], row["after_FT_%"]),
                        textcoords="offset points", xytext=(0, 8),
                        fontsize=7.5, ha="center", color=color)

        if prune_type == "structured":
            red_patch = mpatches.Patch(color="red", alpha=0.2, label="⛔ Quant fail")
            ax.legend(handles=ax.get_legend_handles_labels()[0] + [red_patch],
                      fontsize=8)
        else:
            ax.legend(fontsize=8)

        ax.set_title(f"{model_name} — {prune_type}")
        ax.set_xlabel("Pruning amount (%)")
        ax.set_ylabel("Val accuracy (%)")
        ax.set_xticks(amounts)
        ax.grid(True, alpha=0.4)

plt.tight_layout()
plot_path = "/content/drive/My Drive/stm32-thesis/pruning_damage_curves.png"
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"✅ Plot saved → {plot_path}")

## Next steps

### ✅ Checkpoints safe for `Pipeline_Quantz`
Use the **deployable checkpoint list** printed above. Only pass `.pth` files where `quant_ok = True`.

### 📊 What to look for in results
- **Unstructured 10–30%**: should recover close to baseline after FT; NSE should hold ≥ 0.98 easily
- **Structured 2–5%**: some delta loss acceptable; NSE is the real gate
- **Structured 10%**: watch for NSE dropping below threshold — if it does, skip it

### 🔢 On parameter counts
`params_total` won't change for structured pruning in this notebook — PyTorch's mask-based pruning zeroes filters but doesn't physically resize layers. `params_nonzero` is the honest number: it reflects real effective capacity and what a sparse-aware deployment tool would compress to.

### 🚫 Do NOT use
Any checkpoint where `quant_ok = False` — pass those directly to the bin, not `Pipeline_Quantz`.